[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/semiconductor_devices/diode.ipynb)

# A p–n Junction

This example computes carrier drift, diffusion, and recombination across a silicon p–n junction as two neutral slabs touch, sweep through forward and reverse bias, and absorb light as a solar cell. Acceptors take electrons from the crystal and leave mobile holes behind, whereas donors supply mobile electrons. Where the two halves meet, electrons diffuse from the n side into the p side and holes diffuse the other way before recombining and releasing the band-gap energy to the lattice as heat. The ionized dopants left behind establish an electric field across the carrier-depleted region that holds the remaining carriers back until each carrier's electrochemical potential is uniform across the bar and the bands bend by the built-in voltage. Forward bias lowers that barrier so current grows exponentially with voltage, reverse bias raises it so current saturates at thermal generation, and absorbed light generates electron-hole pairs that the junction field sweeps apart as a photocurrent opposing forward injection.

Spacetime along the bar is weighed by a null point weight `w` and joined with null quantity directions for electrons, holes, heat, and light, where the term ***parcel*** here means an extensive state combining carrier counts, charge along `time`, heat, or photon energy, so an electron is a unit electron count with negative charge and a hole is a unit hole count with positive charge. A single potential plane at each vertex holds the electric potential and both chemical potentials, so pairing `potentials & electron` reads the electron quasi-Fermi level directly, wedging with the spatial gradient forms one combined drift-diffusion drive `-(gradient ^ potentials)` whose commutator `drive.commutator(electron)` sweeps each carrier along its drift, and pulling fluxes back through `(gradient ^ Potentials).adjugate()` gathers edge fluxes into dual-cell outflows. Recombination acts along a single reaction direction `gap * heat - (electron + hole)` orthogonal to the energy plane, while solar absorption reverses that transition using photon energy from `light` bounded by the Carnot factor of the solar temperature; the companion notebooks extend the same formulation to [a planar MOSFET](mosfet.ipynb) and [solar-cell finger spacing](solar_cell.ipynb).

In vector-calculus drift–diffusion notation, the electrostatic potential $\psi$ and carrier densities $n, p$ satisfy three separate equations with opposite charge signs for drift and diffusion, $-\partial_x(\epsilon\partial_x\psi) = q(p - n + N_D^+ - N_A^-)$, $J_n = -q\mu_n n\partial_x\psi + q D_n\partial_x n$, and $J_p = -q\mu_p p\partial_x\psi - q D_p\partial_x p$, linked through the Einstein relations $D_{n,p} = (k_B T / q)\mu_{n,p}$ and separate continuity equations $\partial_t n = \frac{1}{q}\partial_x J_n - k(np - n_i^2) + G$ and $\partial_t p = -\frac{1}{q}\partial_x J_p - k(np - n_i^2) + G$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from examples.thermodynamics.semiconductor_devices import render, scenarios
from examples.util.animation import show_states
from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag

# Projective bar spacetime weighed by w0, joined with null quantity directions for electrons, holes, heat, and light.
ga = Algebra("x+t-w0") * Algebra("n0p0q0l0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
Spatial = Plane.derive.within(Algebra("x+"))                                  # Vector+
Potentials = Plane.derive.within(Algebra("t-n0p0"))                           # Vector+
Drive = ga.gatype(Spatial ^ Potentials)                                       # Bivector+
# Unit charge along time, charged carrier parcels, and neutral heat and photon energy parcels.
time = -mv.t.dual()                                                           # [] Point-
electron = mv.n.dual() - time                                                 # [] Point-
hole = mv.p.dual() + time                                                     # [] Point-
heat = mv.q.dual()                                                            # [] Point-
light = mv.l.dual()                                                           # [] Point-
pairs = electron + hole                                                       # [] Point-
carriers = stack((electron, hole))                                            # [2] Point-
counts = stack((mv.n.dual(), mv.p.dual()))                                    # [2] Point-
count_planes = stack((mv.n, mv.p))                                            # [2] Plane+
charges = carriers - counts                                                   # [2] Point-
# Span of time and every material quantity along which purely spatial fields stay constant.
unread = time & counts[0] & counts[1] & heat & light                          # [] Span-

## Silicon bar and simplex gradient

A chain of uniform segments represents a room-temperature silicon bar doped equally with acceptors on the left half and donors on the right half. Lengths are measured in the doping Debye length, energies and potentials in thermal energy units, carrier densities in units of the doping density, and times in the Debye electron diffusion time; in these units the intrinsic carrier density is one millionth of the doping density and the silicon band gap is 43.6 thermal units. Each vertex is placed as a unit-weight point. The term ***simplex gradient*** here means the spatial covector at each endpoint of an edge whose pairing with a displacement along the bar gives the linear change of that endpoint's interpolation weight across the edge. Joining the opposite endpoint with `unread`, normalizing by its pairing with the endpoint itself, and projecting onto `Spatial` yields the endpoint simplex gradients; assembling them over edges gives `gradient`, and pulling back `-(gradient ^ Potentials)` via `.adjugate()` preserves pairing with the potential plane, `potentials & outflow(flux) == drive_map(potentials) & flux`, gathering edge fluxes into dual-cell outflows.

In [ ]:
# Room-temperature silicon intrinsic density, band gap in thermal units, carrier mobilities, and permittivity.
intrinsic, gap = 1e-6, 43.6
mobilities, permittivity = np.array([1.0, 1 / 3]), 1.0

In [ ]:
topology, vertices, doping, ends = scenarios.bar()
V, E = topology.n_elements
dopants = (time * doping).field()                                             # Point-[V]
held_indices, free_indices = scenarios.contact_partition([ends], V)
held_reading = topology.reading(held_indices, 0)                              # Scalar+[V_held <- V]
free_reading = topology.reading(free_indices, 0)                              # Scalar+[V_free <- V]
corners = topology.incidence[1, 0] * vertices                                 # [2] Point-[E]
opposite = (topology.off[1, 0] * vertices)[0]                                 # [2] Point-[E]
# Joining the opposite endpoint with unread vanishes there; normalizing at the corner and projecting onto Spatial
# extracts each endpoint's simplex gradient.
gradients = ((opposite & unread) / (opposite & unread & corners)).cast(Spatial)   # [2] Spatial+[E]
lengths = (corners[0] & corners[1] & unread).norm()                           # Scalar+[E]
corner_mean = topology.incidence[1, 0].mean(axis=0)                           # Scalar+[E <- V]
dual_lengths = corner_mean.adjugate() * lengths                               # Scalar+[V]
gradient = (spdiag(gradients) * topology.incidence[1, 0]).sum(axis=0)         # Spatial+[E <- V]
drive_map = -(gradient ^ Potentials)                                          # (Drive+ <- Potentials+)[E <- V]
outflow = drive_map.adjugate()                                                # (Point- <- Flux-)[V <- E]

In [ ]:
def carrier_densities(potentials: Potentials) -> Scalar:
    """Boltzmann's law relates the electron and hole densities (`[2] Scalar+[V]`) to the chemical potentials."""
    return intrinsic * (potentials & counts).exp()


def energy_plane(potentials: Potentials) -> Plane:
    """Electrostatic potential and half-gap offsets combine with thermal and photon directions into the energy covector
    field (`Plane+[V]`)."""
    return (potentials & time) * mv.t + gap / 2 * (mv.n + mv.p) + mv.q + mv.l

In [ ]:
def band_energies(potentials: Potentials) -> Scalar:
    """Pairing the energy and potential planes with an electron and a hole gives the conduction band edge, valence band
    edge, and electron and hole quasi-Fermi levels (`[4] Scalar+[V]`)."""
    energies = energy_plane(potentials)                                       # Plane+[V]
    return stack((energies & electron, -(energies & hole), potentials & electron, -(potentials & hole)))


chemical = scenarios.neutral_potentials(doping)                               # [2, V]
slab = (mv.n * chemical[0] + mv.p * chemical[1]).retyped(Potentials).field()  # Potentials+[V]
window = (-26.0, 60.0)
render.bands_figure(vertices, band_energies(slab), carrier_densities(slab), intrinsic, window);

Before the two halves interact, the conduction band edge above and valence band edge below lie flat across the p side on the left and the n side on the right. Logarithmic blue and red shading marks electron and hole densities from intrinsic level up to the doping level, while alternating blue and red dashes show the coinciding carrier quasi-Fermi levels within each neutral half, separated across the step junction by the built-in voltage.

## Junction formation

Closing a wire between the two ends equalizes their Fermi levels and holds both endpoints as ohmic contacts while the interior relaxes by implicit steps of `dt`. At each step, dual-cell carrier storage, edge drift-diffusion outflows, and mass-action recombination balance simultaneously, while Gauss's law supplies the independent charge balance along `time` where electron and hole conservation coincide. Newton's tangent map combines carrier capacitance, recombination response, and simplex conductances and permittivity pulled back through `outflow`. Running the cell below takes about two seconds.

In [ ]:
dt, n_frames, frame_duration_ms = 0.7, 100, 40
lifetime = 2.0
rate_constant = 1 / lifetime
limit, tolerance = 6.0, 0.1

In [ ]:
unit_conductances = mobilities * Drive.commutator(carriers).regressive(carriers)   # [2] Flux- <- Drive+
displacement = permittivity * Drive.commutator(time).regressive(time)         # [] Flux- <- Drive+


def potential_frames(potentials: Potentials, contacts: Potentials, generations: np.ndarray,
                     dt: float) -> Iterator[Potentials]:
    """Yields the initial vertex potentials (`Potentials+[V]`) and the implicit Newton solution after each step `dt`
    under held `contacts` (`[n_frames] Potentials+[V_held]`) and uniform pair `generations` (`[n_frames]`)."""
    yield potentials
    for contact, generation in zip(contacts, generations):
        previous_parcels = (carrier_densities(potentials) * carriers).sum(axis=0)   # Point-[V]
        potentials = free_reading.adjugate() * (free_reading * potentials) + held_reading.adjugate() * contact
        while True:
            densities = carrier_densities(potentials)                         # [2] Scalar+[V]
            parcels = (densities * carriers).sum(axis=0)                      # Point-[V]
            drive = drive_map(potentials)                                     # Drive+[E]
            conductance = ((corner_mean * densities) * unit_conductances).sum(axis=0)   # (Flux- <- Drive+)[E]
            rates = rate_constant * (densities[0] * densities[1] - intrinsic ** 2) - generation   # Scalar+[V]
            charge = (densities * charges).sum(axis=0) + dopants              # Point-[V]
            gauss = outflow(lengths * displacement(drive)) - dual_lengths * charge   # Point-[V]
            residual = (dual_lengths * (parcels - previous_parcels) / dt + outflow(lengths * conductance(drive))
                        + dual_lengths * rates * pairs + gauss)               # Point-[V]
            capacitance = dual_lengths * (densities * (carriers / dt - charges) * (Potentials & counts)).sum(axis=0)
            drive_response = (lengths * (conductance + displacement))(drive_map)
            density_response = (spdiag(lengths * unit_conductances(drive)) * corner_mean
                                * spdiag(densities * (Potentials & counts))).sum(axis=0)
            pair_response = dual_lengths * rate_constant * densities[0] * densities[1] * (pairs * (Potentials & pairs))
            jacobian = capacitance + pair_response + outflow(drive_response + density_response)
            # Scaling each carrier equation by local density keeps minority and majority balances equally weighted.
            balance_weights = time * (mv.t & Point) + (counts * (count_planes & Point) / densities).sum(axis=0)
            free_jacobian = free_reading * balance_weights(jacobian) * free_reading.adjugate()
            step_change = -(free_reading.adjugate() * free_jacobian.solve(free_reading * balance_weights(residual)))
            chemical_shift = (counts & step_change).abs().sum(axis=0)         # Scalar+[V]
            potentials = potentials + step_change / (1 + chemical_shift / limit)
            if chemical_shift.max() < tolerance:
                break
        yield potentials

In [ ]:
built_in = chemical[1, 0] + chemical[0, -1]
wired_contacts = held_reading * slab + (mv.t * scenarios.bar_voltages(-built_in)).field()   # Potentials+[2]
formation_states = show_states(
    potential_frames(slab, np.ones(n_frames) * wired_contacts, np.zeros(n_frames), dt),
    render.formation_animation, vertices, band_energies, carrier_densities, intrinsic, window,
    name="diode_forming", duration_ms=frame_duration_ms)

When the end wire closes, the built-in voltage drop initially tilts the bands uniformly along the bar before majority-carrier dielectric relaxation concentrates the electric field at the junction. Electrons and holes spill across the metallurgical boundary and split the quasi-Fermi levels until recombination exhausts the injected excess, leaving a pale carrier-depleted space-charge region whose band bending aligns a single flat Fermi level from end to end.

## Current under forward and reverse bias

Raising the p-contact potential sweeps the junction from zero into forward bias, back through zero into reverse bias, and back to zero through steady states at `dt = np.inf`. In steady state the total charge current is uniform from segment to segment and equals the time component of the dual-cell outflow entering the grounded n contact, `t & outflow(lengths * flux)`. Running the cell below takes about three seconds.

In [ ]:
biases = scenarios.bias_sweep()
n_sweep_frames = len(biases)
volt, current_window = 1 / 0.02585, (-1.35e-4, 1.35e-4)

In [ ]:
sweep_contacts = wired_contacts + (mv.t * scenarios.bar_voltages(biases)).field()   # [n_sweep_frames] Potentials+[2]
end_rise = topology.reading(ends[1:], 0) - topology.reading(ends[:1], 0)      # Scalar+[1 <- V]
n_contact = topology.reading(ends[1:], 0)                                     # Scalar+[1 <- V]


def terminal_readings(potentials: Potentials) -> tuple[Scalar, Scalar]:
    """Contact quasi-Fermi difference and n-end charge inflow determine the terminal voltage and current (`[] Scalar+`
    each)."""
    conductance = ((corner_mean * carrier_densities(potentials)) * unit_conductances).sum(axis=0)   # (Flux- <- Drive+)[E]
    net_outflow = outflow(lengths * conductance(drive_map(potentials)))       # Point-[V]
    return (end_rise * (potentials & electron)).sites.sum(), -(n_contact * (mv.t & net_outflow)).sites.sum()


dark_states = show_states(
    potential_frames(formation_states[-1], sweep_contacts, np.zeros(n_sweep_frames), np.inf),
    render.sweep_animation, vertices, band_energies, carrier_densities, intrinsic, terminal_readings, window,
    (biases.min(), biases.max()), current_window, volt, render.dark_colour, [], name="diode_dark",
    duration_ms=frame_duration_ms)

Under forward bias the potential barrier shrinks, the electron and hole quasi-Fermi levels split across the depletion region as minority carriers inject into opposite halves and decay over a diffusion length, and the terminal current rises exponentially with each thermal volt (roughly tenfold every 60 mV near 0.4 V). Under reverse bias the barrier grows, the depletion zone widens, and the reverse current saturates at thermal generation inside the depleted region, invisibly small on the forward-current scale.

## Illuminated junction and photovoltaic response

Optical absorption generates electron-hole pairs along `pairs - gap * light`, reversing recombination while keeping total parcel energy conserved. Because solar photons arrive at the sun's temperature relative to the lattice, photon free energy carries the Carnot factor `1 - 1 / sun`, and absorption produces positive entropy whenever the quasi-Fermi splitting stays below `gap * (1 - 1 / sun)`. Illumination rises from dark to full intensity at short circuit before repeating the full voltage sweep. Running the cell below takes about three seconds.

In [ ]:
generation_rate = 1e-5
sun = 5800 / 300
lit_biases, shares = scenarios.lit_bias_sweep()
n_lit_frames = len(lit_biases)
generations = generation_rate * shares                                        # [n_lit_frames]

In [ ]:
lit_contacts = wired_contacts + (mv.t * scenarios.bar_voltages(lit_biases)).field()   # [n_lit_frames] Potentials+[2]
lit_states = show_states(
    potential_frames(dark_states[-1], lit_contacts, generations, np.inf), render.sweep_animation,
    vertices, band_energies, carrier_densities, intrinsic, terminal_readings, window, (biases.min(), biases.max()),
    current_window, volt, render.lit_colour, [dark_states], name="diode_lit", duration_ms=frame_duration_ms)

Photogenerated minority carriers raise the electron quasi-Fermi level on the p side and lower the hole quasi-Fermi level on the n side. At short circuit the built-in electric field sweeps those pairs apart into a reverse photocurrent that shifts the orange illuminated curve downward below the faint dark curve, with slightly stronger collection in reverse bias where the wider depletion zone captures more pairs. Between zero voltage and the open-circuit intercept where forward injection balances collection, current opposes voltage and the illuminated diode delivers electrical power with quasi-Fermi splitting bounded well below the 41-thermal-unit Carnot ceiling.

In photovoltaic circuit notation, superposition shifts the Shockley diode curve by the short-circuit photocurrent $J_L$ to $J = J_0(e^{qV / k_B T} - 1) - J_L$ with open-circuit voltage $V_{\mathrm{oc}} = (k_B T / q)\ln(1 + J_L / J_0)$ bounded thermodynamically by $q V_{\mathrm{oc}} \le E_g(1 - T / T_\odot)$.

In [ ]:
# checks
recombination = gap * heat - pairs                                            # [] Point-
absorption = pairs - gap * light                                              # [] Point-
# Both reaction transitions conserve parcel energy identically.
for state in (formation_states[-1], lit_states[-1]):
    assert ((energy_plane(state) & recombination).batch().abs() < 1e-10).all()
    assert ((energy_plane(state) & absorption).batch().abs() < 1e-10).all()
# Forward bias grows exponentially at one e-fold per thermal volt, while reverse bias saturates.
voltages, currents = (stack(readings) for readings in zip(*(terminal_readings(state) for state in dark_states)))
forward = (voltages > 4) & (voltages < 14)
log_currents = currents[forward].log()
forward_voltages = voltages[forward]
slope = (log_currents[1:] - log_currents[:-1]) / (forward_voltages[1:] - forward_voltages[:-1])
assert ((slope > 0.9) & (slope < 1.1)).all()
reverse = -currents[voltages < -3]
assert reverse.max() < 1.15 * reverse.min()
# Transport, recombination, and optical absorption produce non-negative entropy across every solved frame.
lit_generations = np.concatenate([[0.0], generations])
all_runs = [(state, 0.0) for state in formation_states + dark_states] + list(zip(lit_states, lit_generations))
transport_rates, recomb_rates, absorb_rates = [], [], []
for state, generation in all_runs:
    densities = carrier_densities(state)                                      # [2] Scalar+[V]
    drive = -(gradient ^ state)                                               # Drive+[E]
    conductance = ((corner_mean * densities) * unit_conductances).sum(axis=0)   # (Flux- <- Drive+)[E]
    entropy_slope = energy_plane(state) - state - (1 - 1 / sun) * mv.l        # Plane+[V]
    rates = rate_constant * (densities[0] * densities[1] - intrinsic ** 2)    # Scalar+[V]
    transport_rates.append((lengths * (drive & conductance(drive))).batch())
    recomb_rates.append((dual_lengths * rates * (entropy_slope & recombination)).batch())
    absorb_rates.append((dual_lengths * generation * (entropy_slope & absorption)).batch())
for production in (stack(transport_rates), stack(recomb_rates), stack(absorb_rates)):
    assert production.min() > -1e-12 * production.max()
# Quasi-Fermi splitting under illumination stays strictly below the solar Carnot bound.
for state in lit_states:
    bands = band_energies(state)                                              # [4] Scalar+[V]
    assert (bands[2] - bands[3]).batch().max() < gap * (1 - 1 / sun)